# 모델 실험 템플릿

팀 공통 실험 노트북입니다. **이 파일은 고치지 말고 복사해서** 씁니다.
각자 모델 하나(`yolo11m.pt` 또는 `yolo26m.pt`)를 맡아 옵티마이저·lr0 등을 조정합니다.

1. `notebooks/exp_<이름>.ipynb`로 복사합니다.
2. 설정 셀의 **내 실험** 부분만 바꿉니다. **팀 공통** 값은 모두 같아야 서로 점수를 비교할 수 있으므로 바꾸지 않습니다.
3. 순서
   1. 로컬(Mac)에서 `SMOKE = True`로 동작 확인 (epoch 1, 학습 사진 10%, wandb 끔)
   2. Colab GPU에서 `SMOKE = False`, `STAGE = "screen"`: fold 0만 학습해 설정을 고릅니다. 다른 설정과는 fold 0 점수끼리 비교합니다.
   3. 고른 설정은 `STAGE = "confirm"`: 5 fold를 모두 학습합니다. fold 0은 screen 결과를 그대로 씁니다. **모델 비교는 이 5 fold 평균·표준편차로** 합니다.
4. 실험 이름은 설정으로 자동으로 만듭니다 (예: `a_yolo26m_AdamW_lr0.001_cos`). 같은 설정을 다시 돌리면 끝난 fold는 건너뛰고, 끊긴 fold는 `last.pt`에서 이어 학습합니다.
5. 결과는 wandb 팀 프로젝트에 `<실험 이름>/<run>` 이름으로 쌓이고, config에 옵티마이저·lr0 등이 남습니다.
   - Colab: 커널 선택 → **Colab** → GPU 런타임. VS Code 탐색기에서 `.env`를 우클릭 → **Upload to Colab** (`DATA_URL`, `WANDB_API_KEY`, `WANDB_ENTITY`).
   - Colab은 `BRANCH`의 코드를 받아 돌리므로, 로컬에서 고친 `.py`는 push한 뒤 실행합니다.
6. 커밋할 때는 출력을 지웁니다.

## 1. 설정

실험마다 **내 실험**과 **실행** 부분만 바꿉니다. 팀 공통 값을 바꿔야 할 일이 생기면 팀에서 정한 뒤 템플릿을 고칩니다.

In [ ]:
# ===== 팀 공통: 바꾸지 않는다 (AGENTS.md 기본값) =====
LABEL = "v1"
SPLIT = "setup30fix_v1_s42"
EPOCHS = 100
IMGSZ = 640
SEED = 42
AUG = "on"  # models/train.py의 AUG_PRESETS
FRACTION = 1.0

# ===== 내 실험 =====
MEMBER = "a"  # 이름이나 이니셜. 실험 이름 맨 앞에 붙는다
MODEL = "yolo26m.pt"  # yolo11m.pt 또는 yolo26m.pt
# SGD, MuSGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp. auto는 lr0를 무시하므로 쓰지 않는다
OPTIMIZER = "AdamW"
LR0 = 1e-3  # None이면 SGD 계열 0.01, 그 밖 0.001
COS_LR = False  # True면 학습률을 cosine 곡선으로 줄인다
BATCH = 16  # 팀 기본 16. GPU 메모리가 부족하면 줄인다
# 그 밖 train() 인자. 괄호 안은 ultralytics 기본값
# lrf(0.01), momentum(0.937), weight_decay(0.0005), warmup_epochs(3.0), patience(100)
TRAIN_KWARGS = {}  # 예: {"weight_decay": 0.001, "patience": 30}
TAG = ""  # 이름이 같아지는 실험을 구분할 짧은 꼬리표 (선택)

# ===== 실행 =====
SMOKE = True  # True면 동작 확인만. Colab GPU 학습은 False
STAGE = "screen"  # screen: fold 0만, confirm: 5 fold 전부
BRANCH = "main"  # Colab에서 받을 브랜치
WANDB = True  # False면 wandb에 기록하지 않는다

## 2. 환경

Colab이면 저장소를 받고 패키지를 설치한 뒤 결과 폴더를 Drive에 둡니다. 로컬에서는 프로젝트 루트를 import 경로에 넣기만 합니다.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()
RUNS_DIR = None  # None이면 프로젝트의 runs/

if IN_COLAB:
    # colab.ipynb 2–4단계와 같다: 저장소 받기, uv.lock 버전 설치, .env 읽기
    import importlib

    REPO_DIR = Path("/content/pill-bbox")
    if REPO_DIR.exists():
        !git -C {REPO_DIR} fetch -q origin {BRANCH}
        !git -C {REPO_DIR} checkout -q {BRANCH}
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q -b {BRANCH} https://github.com/def-uk/pill-bbox.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if not Path("~/.local/bin/uv").expanduser().exists():
        !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = f"{Path('~/.local/bin').expanduser()}:{os.environ['PATH']}"
    # torch는 lock에 적힌 CUDA 12.6 빌드로 설치한다
    req = "/tmp/requirements.txt"
    torch_index = "--extra-index-url https://download.pytorch.org/whl/cu126"
    strategy = "--index-strategy unsafe-best-match"
    !uv export -q --frozen --no-dev --no-hashes --no-emit-project -o {req}
    !uv pip install -q --python {sys.executable} -r {req} {torch_index} {strategy}

    # Colab은 시작할 때 numpy 등을 미리 import한다. 설치로 버전이 바뀌면 메모리의 옛 버전과
    # 새로 깐 패키지(sklearn 등)가 섞여 import가 깨지므로 커널을 재시작한다.
    # 재시작 뒤 이 셀부터 다시 실행하면 설치는 이미 끝나 있어 바로 넘어간다.
    import signal
    from importlib.metadata import version

    dists = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "sklearn": "scikit-learn"}
    dists |= {"torch": "torch", "matplotlib": "matplotlib", "PIL": "pillow"}
    stale = [
        name
        for name, dist in dists.items()
        if name in sys.modules
        and getattr(sys.modules[name], "__version__", version(dist)) != version(dist)
    ]
    if stale:
        print(f"설치로 {stale} 버전이 바뀌어 커널을 재시작합니다. 맨 위 셀부터 다시 실행하세요.")
        sys.stdout.flush()
        os.kill(os.getpid(), signal.SIGKILL)
    sys.path.insert(0, str(REPO_DIR / "notebooks"))
    importlib.invalidate_caches()

    from dotenv import load_dotenv

    load_dotenv("/content/.env")  # 이미 설정된 환경 변수가 우선한다
    for name in ["DATA_URL", "WANDB_API_KEY", "WANDB_ENTITY"]:
        if not os.getenv(name):
            try:
                from google.colab import userdata

                os.environ[name] = userdata.get(name)
            except Exception:
                pass
    if not os.getenv("WANDB_API_KEY"):
        os.environ["WANDB_MODE"] = "disabled"
        print("WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.")

    # 결과를 Drive에 두어야 런타임이 끊겨도 남는다
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        RUNS_DIR = Path("/content/drive/MyDrive/pill-bbox/runs")
    except Exception as e:
        print(f"Drive 연결 실패({e}). 결과가 런타임에만 남으니 끊기기 전에 내려받으세요.")
    !git branch --show-current && git log --oneline -1

# 프로젝트 루트를 import 경로에 넣는다
import nb_data  # noqa: E402, F401

## 3. 실험 준비

설정을 검사하고 실험 이름을 만듭니다. 같은 학습을 터미널에서 다시 돌리는 `model_main` 명령을 출력하고, 설정은 `<RUNS_DIR>/<exp>/notebook_config.json`에 남깁니다.

In [ ]:
import json
from datetime import datetime

from data_process.split import N_FOLDS
from models.train import default_lr0
from src.config import PROJECT_ROOT, get_device
from src.config import RUNS_DIR as DEFAULT_RUNS_DIR

# 가중치 이름(yolo11m.pt)은 현재 폴더에 받아지므로 루트로 옮겨 팀 공통 위치에 둔다
os.chdir(PROJECT_ROOT)
RUNS_DIR = RUNS_DIR or DEFAULT_RUNS_DIR
DEVICE = get_device()

if not MEMBER:
    raise ValueError("MEMBER에 이름이나 이니셜을 적으세요.")
if MODEL not in ["yolo11m.pt", "yolo26m.pt"]:
    raise ValueError(f"후보 모델은 yolo11m.pt, yolo26m.pt입니다: {MODEL}")
if OPTIMIZER == "auto":
    raise ValueError("auto는 lr0를 무시합니다. 옵티마이저를 직접 고르세요 (예: AdamW).")
if STAGE not in ["screen", "confirm"]:
    raise ValueError(f"STAGE는 screen 또는 confirm입니다: {STAGE}")
lr0 = default_lr0(OPTIMIZER) if LR0 is None else LR0

# 결과 폴더 이름에는 lr0 등이 없으므로, 내 실험 설정을 모두 실험 이름에 넣어 구분한다
parts = [MEMBER, Path(MODEL).stem, OPTIMIZER, f"lr{lr0:g}"]
parts += ["cos"] if COS_LR else []
parts += [] if BATCH == 16 else [f"b{BATCH}"]
parts += [
    f"{key}{value:g}" if isinstance(value, float) else f"{key}{value}"
    for key, value in sorted(TRAIN_KWARGS.items())
]
parts += [TAG] if TAG else []
exp_name = "_".join(parts)

# 동작 확인이면 짧은 값을 쓰고, 실험 이름에 시각을 붙여 이전 확인과 겹치지 않게 한다
if SMOKE:
    exp, epochs, fraction, folds = f"smoke_{exp_name}_{datetime.now():%m%d_%H%M}", 1, 0.1, [0]
    os.environ["WANDB_MODE"] = "disabled"
else:
    if DEVICE != "cuda":
        raise RuntimeError(f"본 학습은 GPU에서 합니다(지금 {DEVICE}). 동작 확인은 SMOKE = True.")
    exp, epochs, fraction = exp_name, EPOCHS, FRACTION
    folds = [0] if STAGE == "screen" else list(range(N_FOLDS))
if not WANDB:
    os.environ["WANDB_MODE"] = "disabled"
elif not SMOKE:
    # 같은 커널에서 앞서 끈 설정(WANDB_MODE=disabled)이 남아 있으면 기록이 조용히 빠진다
    if not os.getenv("WANDB_API_KEY"):
        raise RuntimeError("WANDB = True인데 WANDB_API_KEY가 없습니다. .env를 Colab에 올리세요.")
    if os.environ.pop("WANDB_MODE", None) == "disabled":
        print("남아 있던 WANDB_MODE=disabled를 지우고 wandb에 기록합니다.")

# 같은 학습을 터미널에서 다시 돌릴 수 있게 model_main 명령으로 남긴다
cmd = [] if WANDB and not SMOKE else ["WANDB_MODE=disabled"]
cmd += ["uv run python -m models.model_main", "--model", MODEL, "--label", LABEL]
cmd += ["--split", SPLIT, "--folds", *map(str, folds), "--epochs", str(epochs), "--exp", exp]
cmd += ["--imgsz", str(IMGSZ), "--batch", str(BATCH), "--seed", str(SEED)]
cmd += ["--fraction", str(fraction), "--aug", AUG, "--optimizer", OPTIMIZER, "--lr0", f"{lr0:g}"]
cmd += ["--cos-lr"] if COS_LR else []
cmd += [] if RUNS_DIR == DEFAULT_RUNS_DIR else ["--runs-dir", str(RUNS_DIR)]
cmd += ["--resume"]
command = " ".join(cmd)

config = {
    "exp": exp,
    "member": MEMBER,
    "stage": STAGE,
    "smoke": SMOKE,
    "wandb": WANDB and not SMOKE,
    "device": DEVICE,
    "model": MODEL,
    "label": LABEL,
    "split": SPLIT,
    "folds": folds,
    "epochs": epochs,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "fraction": fraction,
    "aug": AUG,
    "optimizer": OPTIMIZER,
    "lr0": lr0,
    "cos_lr": COS_LR,
    "train_kwargs": TRAIN_KWARGS,
    "command": command,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
exp_dir = RUNS_DIR / exp
exp_dir.mkdir(parents=True, exist_ok=True)
(exp_dir / "notebook_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"실험 {exp} ({STAGE}, fold {folds}), 장치 {DEVICE}")
print(f"결과 {exp_dir}")
print(command)
if TRAIN_KWARGS:
    print(f"추가 train() 인자 (CLI에 없음): {TRAIN_KWARGS}")

## 4. 데이터

데이터는 `data_process.data_main`으로만 만듭니다. fold마다 train/val 사진 수입니다.

In [ ]:
from nb_data import ensure_pipeline, load_folds

# 라벨·fold·data.yaml이 없으면 data_main을 실행해 만든다 (처음 약 25초, 원본이 없으면 받기부터)
ensure_pipeline(LABEL, [SPLIT])
load_folds(SPLIT).groupby(["fold", "split"]).size().unstack()

## 5. 학습과 평가

`best.pt`는 검증 mAP75-95가 가장 높은 epoch입니다. 런타임이 끊기면 위에서부터 다시 실행하면 이어서 학습합니다.

In [ ]:
from models.evaluate import MAIN_METRIC, evaluate_fold, summarize_exp
from models.train import train

# 실험 이름이 설정에서 나오므로 같은 이름 = 같은 설정이다. 그래서 항상 resume으로 돌린다:
# 끝난 fold(screen의 fold 0 등)는 건너뛰고, 끊긴 fold는 last.pt에서 이어 간다.
for fold in folds:
    save_dir = train(
        MODEL,
        LABEL,
        SPLIT,
        fold,
        epochs=epochs,
        exp=exp,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        fraction=fraction,
        aug=AUG,
        optimizer=OPTIMIZER,
        lr0=lr0,
        cos_lr=COS_LR,
        runs_dir=RUNS_DIR,
        resume=True,
        **TRAIN_KWARGS,
    )
    _, summary = evaluate_fold(
        save_dir / "weights" / "best.pt",
        LABEL,
        SPLIT,
        fold,
        exp=exp,
        imgsz=IMGSZ,
        batch=BATCH,
        runs_dir=RUNS_DIR,
    )
    print(f"{save_dir.name}: {MAIN_METRIC} {summary[MAIN_METRIC]:.4f}")

## 6. 결과 요약

다른 사람·다른 설정과 비교할 때 screen은 fold 0끼리, 모델 비교는 confirm의 5 fold 평균·표준편차로 합니다.

In [ ]:
# fold 평균·표준편차. <exp>/results/summary.csv에도 저장된다. screen은 fold 하나라 std가 비어 있다.
summary = summarize_exp(exp, RUNS_DIR)
cols = [("n_folds", "count"), (MAIN_METRIC, "mean"), (MAIN_METRIC, "std")]
cols += [("map50", "mean"), ("map50_95", "mean")]
summary[cols].round(4)

## 7. 학습 곡선

fold별 매 epoch 학습·검증 loss(윗줄)와 검증 지표 precision·recall·mAP(아랫줄)입니다. 모두 꺾은선입니다. 같은 그림이 run 폴더의 `results.png`로도 저장됩니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator

from models.train import MAIN_METRIC_KEY

# 검증 지표. best.pt 별표를 맨 끝 칸에 찍으므로 MAIN_METRIC_KEY를 마지막에 둔다
METRIC_KEYS = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    MAIN_METRIC_KEY,
]

# ultralytics가 매 epoch 쓰는 results.csv를 fold별로 읽는다 (run 폴더 이름 끝이 _fold<k>)
histories = {}
for path in sorted(exp_dir.glob("*/results.csv")):
    history = pd.read_csv(path)
    history.columns = history.columns.str.strip()
    histories[path.parent.name.rsplit("_", 1)[-1]] = history

if not histories:
    print(f"{exp_dir}에 results.csv가 없습니다. 학습 셀을 먼저 실행하세요.")
else:
    first = next(iter(histories.values()))
    loss_names = [c.removeprefix("train/") for c in first.columns if c.startswith("train/")]
    n_cols = max(len(loss_names), len(METRIC_KEYS))
    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 7), squeeze=False)
    for i, (fold, history) in enumerate(histories.items()):
        color, epoch = f"C{i}", history["epoch"]
        # 윗줄: loss. 실선은 train, 점선은 val. val이 오르는데 train만 내려가면 과적합이다
        for ax, name in zip(axes[0], loss_names, strict=False):
            ax.plot(epoch, history[f"train/{name}"], color=color, marker=".", label=f"{fold} train")
            ax.plot(epoch, history[f"val/{name}"], color=color, linestyle="--", label=f"{fold} val")
            ax.set_title(name)
        # 아랫줄: 검증 지표. 별은 best.pt를 고른 epoch (mAP75-95 최고)
        for ax, key in zip(axes[1], METRIC_KEYS, strict=False):
            ax.plot(epoch, history[key], color=color, marker=".", label=fold)
            ax.set_title(key.removeprefix("metrics/").removesuffix("(B)"))
        best = history[MAIN_METRIC_KEY].idxmax()
        axes[1, len(METRIC_KEYS) - 1].plot(
            epoch[best], history.loc[best, MAIN_METRIC_KEY], "*", color=color, markersize=14
        )
    for ax in axes.ravel():
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    for ax in [*axes[0, len(loss_names) :], *axes[1, len(METRIC_KEYS) :]]:
        ax.set_visible(False)
    axes[0, 0].legend(fontsize=7)
    axes[1, 0].legend(fontsize=7)
    fig.suptitle(exp)
    fig.tight_layout()
    plt.show()

## 8. 제출 csv

`best.pt`로 test 사진을 예측해 대회 제출 형식(`annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`)으로 저장합니다. `image_id`는 test 파일 이름의 숫자, `category_id`는 원본 JSON의 id, bbox는 원본 사진 픽셀 기준입니다. 파일은 `<RUNS_DIR>/<exp>/results/`에 생깁니다.

In [ ]:
from models.evaluate import make_submission
from models.train import run_dir

SUBMIT_FOLD = folds[0]  # 제출에 쓸 fold의 best.pt. screen은 fold 0뿐이다
# 이 점수 미만의 박스는 버린다. 대회는 mAP@[0.75:0.95] 채점이라 낮게 둬도 점수가 떨어지지 않는다
SUBMIT_CONF = 0.001

weights = run_dir(MODEL, SPLIT, SUBMIT_FOLD, exp, AUG, OPTIMIZER, RUNS_DIR) / "weights" / "best.pt"
if not weights.is_file():
    raise FileNotFoundError(f"{weights}가 없습니다. 학습 셀을 먼저 실행하세요.")
submission, submission_path = make_submission(weights, LABEL, exp, IMGSZ, SUBMIT_CONF, RUNS_DIR)
print(f"사진 {submission['image_id'].nunique()}장, 박스 {len(submission)}개")
print(f"저장 {submission_path}")
submission.head()

## 실험 기록

- **실험 이름 / 단계**: (설정 셀 출력의 이름, screen 또는 confirm)
- **이 설정을 고른 이유**:
- **명령**: 실험 준비 셀이 출력한 `uv run python -m models.model_main ...`
- **결과**: screen이면 fold 0 mAP75-95, confirm이면 5 fold 평균 ± 표준편차
- **결론 / 다음 실험**: 